# 07. Neural Networks from Scratch

PyTorch의 고수준 Trainer 없이 **손실 함수, 선형/로지스틱 회귀, MLP 학습**을 직접 구성합니다. 수식을 tensor 연산으로 옮기는 감각과 모델이 학습되지 않을 때 확인할 기준을 만듭니다.

예상 시간: 90~120분 · 실행 장치: CUDA → MPS → CPU 자동 선택 · 모든 데이터는 셀에서 합성

## 학습 지도

1. MSE와 선형 회귀: forward/backward/step의 최소 단위
2. sigmoid와 binary cross entropy: 확률적 이진 분류
3. MLP와 비선형성: 선형 모델이 못 푸는 XOR
4. activation과 initialization: 신호·gradient의 흐름
5. dropout과 normalization: 학습/추론 동작 차이
6. 작은 배치 과적합: 가장 강력한 학습 파이프라인 점검법

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math
import pandas as pd
import torch
from torch import nn
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
torch.set_default_device(device)

print("torch:", torch.__version__)
print(ACCELERATOR.summary())

## 1. MSE와 선형 회귀

회귀에서는 예측 $ŷ$과 정답 $y$의 거리를 줄입니다.

$$\mathrm{MSE}(y,\hat{y}) = \frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2$$

작은 toy data에서는 정답 관계를 알고 있으므로 학습 결과를 검산할 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.randn,torch.cat -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
x_reg = torch.linspace(-2, 2, 64).unsqueeze(1)
noise = 0.15 * torch.randn(x_reg.shape)
y_reg = 3.0 * x_reg - 0.5 + noise

print("x/y shape:", x_reg.shape, y_reg.shape)
print(torch.cat([x_reg[:3], y_reg[:3]], dim=1))
assert x_reg.shape == y_reg.shape == (64, 1)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,F.mse_loss,Tensor.item,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>F.mse_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.mse_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 제곱 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 모든 원소의 제곱 오차를 평균냅니다.
- **직관:** 큰 오차를 제곱해 더 강하게 벌점하므로 회귀와 reconstruction에 자주 씁니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.mse_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def mse_manual(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    assert prediction.shape == target.shape
    return ((prediction - target) ** 2).mean()


trial_prediction = torch.tensor([[1.0], [2.0], [3.0]])
trial_target = torch.tensor([[1.5], [1.5], [2.5]])
manual = mse_manual(trial_prediction, trial_target)
builtin = nn.functional.mse_loss(trial_prediction, trial_target)
print("manual/builtin:", manual.item(), builtin.item())
assert torch.allclose(manual, builtin)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.zeros,Tensor.backward,torch.no_grad -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# nn.Linear 없이 parameter 두 개로 y_hat = xw + b를 구현합니다.
torch.manual_seed(SEED)
weight = torch.randn(1, 1, requires_grad=True)
bias = torch.zeros(1, requires_grad=True)
learning_rate = 0.08

for step in range(151):
    prediction = x_reg @ weight + bias
    loss = mse_manual(prediction, y_reg)
    loss.backward()

    with torch.no_grad():
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
    weight.grad = None
    bias.grad = None

    if step % 50 == 0:
        print(f"step={step:3d} loss={loss.item():.5f}")

print("learned weight/bias:", weight.item(), bias.item())
assert abs(weight.item() - 3.0) < 0.1
assert abs(bias.item() + 0.5) < 0.1

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,torch.optim.SGD,Module.parameters,Optimizer.zero_grad,Optimizer.step -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>torch.optim.SGD(params, lr, momentum=0, weight_decay=0, ...)</code></summary>

#### `torch.optim.SGD(params, lr, momentum=0, weight_decay=0, ...)`

- **역할:** gradient 방향으로 파라미터를 이동하는 SGD optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률·momentum 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 고전 CNN과 최적화 기초 실험의 update 규칙을 명시합니다.
- **주의점:** momentum·weight decay·Nesterov 설정에 따라 실제 update 식이 달라집니다.

##### 관련 수식과 코드 연결

$$
g_t=\nabla_\theta L_t,\qquad v_t=\mu v_{t-1}+g_t,\qquad\theta_{t+1}=\theta_t-\eta v_t
$$

- **기호:** $\eta$는 학습률, $\mu$는 momentum 계수입니다. $\mu=0$이면 기본 SGD입니다.
- **수식 → 코드:** `momentum`이 0이면 현재 gradient만, 양수이면 이전 update 방향도 함께 사용합니다.
- **직관:** 손실이 가장 빠르게 증가하는 gradient의 반대 방향으로 parameter를 이동합니다.
- **shape 확인:** 각 momentum buffer는 대응 parameter와 같은 shape입니다.
- **공식 문서:** [torch.optim.SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# 같은 계산을 nn.Linear + optimizer로 다시 표현합니다.
torch.manual_seed(SEED)
linear_model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.08)

for _ in range(151):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.mse_loss(linear_model(x_reg), y_reg)
    loss.backward()
    optimizer.step()

print("nn.Linear weight/bias:", linear_model.weight.item(), linear_model.bias.item())
print("parameter names:", [name for name, _ in linear_model.named_parameters()])
assert linear_model(x_reg).shape == y_reg.shape

## 2. Logistic regression

선형 점수(logit)를 sigmoid로 확률로 바꿉니다.

$$z=xW+b, \qquad p(y=1|x)=\sigma(z)=\frac{1}{1+e^{-z}}$$

학습에는 확률을 먼저 만든 뒤 log를 취하기보다 `binary_cross_entropy_with_logits`를 쓰는 것이 수치적으로 안정적입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.ones,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
torch.manual_seed(SEED)
class0 = torch.randn(60, 2) * 0.45 + torch.tensor([-1.0, -0.8])
class1 = torch.randn(60, 2) * 0.45 + torch.tensor([1.0, 0.8])
x_binary = torch.cat([class0, class1])
y_binary = torch.cat([torch.zeros(60), torch.ones(60)]).unsqueeze(1)

permutation = torch.randperm(len(x_binary))
x_binary, y_binary = x_binary[permutation], y_binary[permutation]
print("features/labels:", x_binary.shape, y_binary.shape)
print("positive rate:", y_binary.mean().item())

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.exp,Tensor.clamp,torch.log,F.binary_cross_entropy_with_logits -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.exp(input)</code></summary>

#### `torch.exp(input)`

- **역할:** 각 원소에 자연지수함수 $e^x$를 적용합니다.
- **입력·반환:** 실수 텐서를 받고 같은 shape의 양수 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit, 확률비, diffusion coefficient를 양수 scale로 바꿉니다.
- **주의점:** 큰 양수는 overflow, 큰 음수는 0에 가까운 underflow를 만들 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=e^{x_i}
$$

- **기호:** $e$는 자연로그의 밑입니다.
- **수식 → 코드:** API가 각 원소를 양수 값으로 지수 변환합니다.
- **직관:** 더하기 차이를 곱셈 비율로 바꾸며 log-domain 값을 원래 scale로 복원합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.exp](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)

</details>

<details>
<summary><code>tensor.clamp(min=None, max=None)</code></summary>

#### `tensor.clamp(min=None, max=None)`

- **역할:** 텐서 값을 지정 범위 안으로 제한합니다.
- **입력·반환:** 하한·상한을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률·분모·양자화 범위를 안전한 구간에 둡니다.
- **주의점:** 경계 밖에서는 gradient가 0이 될 수 있어 학습 영향에 주의합니다.

##### 관련 수식과 코드 연결

$$
y=\min\!\left(\max(x,\ell),u\right)
$$

- **기호:** $\ell$은 하한, $u$는 상한입니다.
- **수식 → 코드:** 코드의 `min`·`max` 바깥 값은 각각 경계값으로 치환됩니다.
- **직관:** 수치적으로 위험한 범위를 잘라내지만 경계 밖의 차이는 모두 잃습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [Tensor.clamp](https://docs.pytorch.org/docs/stable/generated/torch.clamp.html)

</details>

<details>
<summary><code>torch.log(input)</code></summary>

#### `torch.log(input)`

- **역할:** 각 원소에 자연로그를 적용합니다.
- **입력·반환:** 양수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 곱을 합으로 바꾸고 log-likelihood를 계산합니다.
- **주의점:** 0은 음의 무한대, 음수는 NaN이므로 작은 epsilon이나 clamp가 필요합니다.

##### 관련 수식과 코드 연결

$$
y_i=\log x_i,\qquad x_i>0
$$

- **기호:** $\log$는 자연로그입니다.
- **수식 → 코드:** API가 곱셈 관계를 덧셈 관계로 바꾸고 작은 확률의 크기를 안정적으로 표현합니다.
- **직관:** 확률 곱을 log-probability 합으로 계산할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같으며 입력은 양수여야 합니다.
- **공식 문서:** [torch.log](https://docs.pytorch.org/docs/stable/generated/torch.log.html)

</details>

<details>
<summary><code>F.binary_cross_entropy_with_logits(input, target, ...)</code></summary>

#### `F.binary_cross_entropy_with_logits(input, target, ...)`

- **역할:** sigmoid와 binary cross entropy를 안정적으로 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{BCE}=-\frac{1}{N}\sum_i\left[y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))\right]
$$

- **기호:** $z_i$는 logit, $y_i\in[0,1]$은 target, $\sigma$는 sigmoid입니다.
- **수식 → 코드:** sigmoid와 BCE를 한 함수에서 log-sum-exp 형태로 안정적으로 계산합니다.
- **직관:** 각 항을 독립적인 이진 선택으로 보고 정답 쪽 확률의 음의 로그를 최소화합니다.
- **shape 확인:** logit과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.binary_cross_entropy_with_logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy_with_logits.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def sigmoid_manual(logits):
    return 1.0 / (1.0 + torch.exp(-logits))


def bce_manual(probabilities, targets, epsilon=1e-7):
    probabilities = probabilities.clamp(epsilon, 1 - epsilon)
    return -(
        targets * probabilities.log() + (1 - targets) * (1 - probabilities).log()
    ).mean()


demo_logits = torch.tensor([[-2.0], [0.0], [2.0]])
demo_targets = torch.tensor([[0.0], [1.0], [1.0]])
probabilities = sigmoid_manual(demo_logits)
manual_bce = bce_manual(probabilities, demo_targets)
stable_bce = nn.functional.binary_cross_entropy_with_logits(demo_logits, demo_targets)

print("probabilities:", probabilities.squeeze().tolist())
print("manual/stable BCE:", manual_bce.item(), stable_bce.item())
assert torch.allclose(manual_bce, stable_bce, atol=1e-6)

In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)  # sigmoid 전의 logit 반환


torch.manual_seed(SEED)
logistic = LogisticRegression(2)
optimizer = torch.optim.SGD(logistic.parameters(), lr=0.25)

for step in range(101):
    optimizer.zero_grad(set_to_none=True)
    logits = logistic(x_binary)
    loss = nn.functional.binary_cross_entropy_with_logits(logits, y_binary)
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        print(f"step={step:3d} BCE={loss.item():.4f}")

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
with torch.inference_mode():
    probabilities = logistic(x_binary).sigmoid()
    predictions = probabilities.ge(0.5)
    accuracy = predictions.eq(y_binary.bool()).float().mean()

print("accuracy:", accuracy.item())
print("first five probabilities:", probabilities[:5].squeeze().tolist())
assert accuracy > 0.95

# TODO: threshold를 0.3, 0.7로 바꾸면 positive 예측 수가 어떻게 달라질까요?
for threshold in [0.3, 0.5, 0.7]:
    print(threshold, int(probabilities.ge(threshold).sum()))

## 3. 왜 MLP가 필요한가: XOR

선형 층을 여러 개 이어도 activation이 없다면 하나의 선형 변환과 같습니다. XOR처럼 하나의 직선으로 나눌 수 없는 문제에는 비선형 activation이 필요합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
xor_x = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
xor_y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

torch.manual_seed(SEED)
xor_linear = nn.Linear(2, 1)
optimizer = torch.optim.Adam(xor_linear.parameters(), lr=0.1)
for _ in range(300):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.binary_cross_entropy_with_logits(xor_linear(xor_x), xor_y)
    loss.backward()
    optimizer.step()

linear_predictions = xor_linear(xor_x).sigmoid().ge(0.5).float()
print("linear predictions:", linear_predictions.squeeze().tolist())
print("linear accuracy:", linear_predictions.eq(xor_y).float().mean().item())

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,Tensor.detach -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 8),
            nn.Tanh(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.network(x)


torch.manual_seed(SEED)
xor_mlp = TinyMLP()
optimizer = torch.optim.Adam(xor_mlp.parameters(), lr=0.05)
for step in range(401):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.binary_cross_entropy_with_logits(xor_mlp(xor_x), xor_y)
    loss.backward()
    optimizer.step()

mlp_probabilities = xor_mlp(xor_x).sigmoid()
mlp_predictions = mlp_probabilities.ge(0.5).float()
print("MLP probabilities:", mlp_probabilities.detach().squeeze().tolist())
print("MLP accuracy:", mlp_predictions.eq(xor_y).float().mean().item())
assert mlp_predictions.eq(xor_y).all()

## 4. Activation과 gradient

- ReLU: 계산이 싸고 깊은 MLP/CNN에서 널리 사용
- GELU: Transformer 계열에서 흔함
- Tanh: 출력 범위가 `[-1,1]`, RNN 내부에서 전통적으로 사용
- Sigmoid: 이진 확률이나 gate에 사용, 큰 절댓값에서 gradient 포화

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame,F.relu,F.gelu,torch.tanh,torch.sigmoid -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>F.gelu(input, approximate=&#x27;none&#x27;)</code></summary>

#### `F.gelu(input, approximate='none')`

- **역할:** 입력을 확률적으로 gate하는 형태의 부드러운 GELU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer FFN의 비선형 변환을 구현합니다.
- **주의점:** 근사 방식에 따라 작은 수치 차이가 생길 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.gelu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.gelu.html)

</details>

<details>
<summary><code>torch.tanh(input)</code></summary>

#### `torch.tanh(input)`

- **역할:** 실수를 -1과 1 사이 값으로 바꿉니다.
- **입력·반환:** 실수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** RNN hidden state나 범위가 제한된 출력을 만듭니다.
- **주의점:** 큰 절댓값에서 포화되어 gradient가 작아질 수 있습니다.

##### 관련 수식과 코드 연결

$$
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

- **기호:** $x$는 실수이고 출력은 -1과 1 사이입니다.
- **수식 → 코드:** API가 activation을 0 중심의 제한된 범위로 압축합니다.
- **직관:** 부호는 유지하면서 큰 값의 영향을 제한합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.tanh](https://docs.pytorch.org/docs/stable/generated/torch.tanh.html)

</details>

<details>
<summary><code>torch.sigmoid(input)</code></summary>

#### `torch.sigmoid(input)`

- **역할:** 실수를 0과 1 사이 값으로 바꿉니다.
- **입력·반환:** logit 텐서를 받고 같은 shape의 확률형 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 이진 확률, gate, attention gate를 표현합니다.
- **주의점:** 큰 절댓값에서는 gradient가 거의 0이 되며 BCE 학습은 logits 버전이 안정적입니다.

##### 관련 수식과 코드 연결

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

- **기호:** $x$는 제한되지 않은 logit이고 $\sigma(x)$는 0과 1 사이입니다.
- **수식 → 코드:** 코드가 logit을 이진 확률이나 gate 값으로 변환합니다.
- **직관:** 0 근처는 민감하고 큰 절댓값에서는 0 또는 1에 포화됩니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sigmoid](https://docs.pytorch.org/docs/stable/generated/torch.sigmoid.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
values = torch.linspace(-3, 3, 7)
activation_table = pd.DataFrame(
    {
        "x": values.tolist(),
        "relu": nn.functional.relu(values).tolist(),
        "gelu": nn.functional.gelu(values).tolist(),
        "tanh": torch.tanh(values).tolist(),
        "sigmoid": torch.sigmoid(values).tolist(),
    }
)
activation_table.round(4)


def activation_gradients(function, points):
    x = points.clone().requires_grad_(True)
    function(x).sum().backward()
    return x.grad


points = torch.tensor([-8.0, -2.0, 0.0, 2.0, 8.0])
gradient_table = pd.DataFrame(
    {
        "x": points.tolist(),
        "sigmoid_grad": activation_gradients(torch.sigmoid, points).tolist(),
        "tanh_grad": activation_gradients(torch.tanh, points).tolist(),
        "gelu_grad": activation_gradients(nn.functional.gelu, points).tolist(),
    }
)
gradient_table.round(6)

## 5. Initialization: 신호 크기를 보존하기

너무 작은 초기값은 신호와 gradient를 사라지게 하고, 너무 큰 값은 폭발시킵니다. Xavier는 tanh 같은 대칭 activation, Kaiming은 ReLU 계열과 잘 맞도록 분산을 조절합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.sqrt -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
torch.manual_seed(SEED)
input_signal = torch.randn(512, 64)


def output_std(weight_std):
    weight = torch.randn(64, 64) * weight_std
    return float((input_signal @ weight).std())


for std in [0.001, 0.02, 1 / math.sqrt(64), 1.0]:
    print(f"weight std={std:.4f} -> output std={output_std(std):.4f}")

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.std -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
xavier_layer = nn.Linear(64, 64, bias=False)
kaiming_layer = nn.Linear(64, 64, bias=False)
nn.init.xavier_uniform_(xavier_layer.weight)
nn.init.kaiming_uniform_(kaiming_layer.weight, nonlinearity="relu")

with torch.inference_mode():
    xavier_output = torch.tanh(xavier_layer(input_signal))
    kaiming_output = torch.relu(kaiming_layer(input_signal))

print("Xavier+tanh std:", xavier_output.std().item())
print("Kaiming+ReLU std:", kaiming_output.std().item())
print("weight shapes:", xavier_layer.weight.shape, kaiming_layer.weight.shape)

## 6. Dropout: train과 eval의 동작 차이

dropout은 학습 중 일부 activation을 0으로 만들고 남은 값을 보정합니다. 평가 시에는 꺼져야 하므로 `model.eval()` 전환이 중요합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Dropout,Module.train,Module.eval,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Dropout(p=0.5)</code></summary>

#### `nn.Dropout(p=0.5)`

- **역할:** 학습 중 일부 activation을 무작위로 0으로 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\frac{m_i}{1-p}x_i,\qquad m_i\sim\operatorname{Bernoulli}(1-p)
$$

- **기호:** $p$는 제거 확률, $m_i$는 학습 중 표본화한 mask입니다.
- **수식 → 코드:** `model.train()`일 때 mask와 $1/(1-p)$ scale을 적용하고 `eval()`에서는 그대로 둡니다.
- **직관:** 일부 activation에 의존하지 못하게 하면서 기댓값은 유지합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
dropout = nn.Dropout(p=0.5)
ones = torch.ones(12)

torch.manual_seed(SEED)
dropout.train()
train_output = dropout(ones)
dropout.eval()
eval_output = dropout(ones)

print("train:", train_output.tolist())
print("eval: ", eval_output.tolist())
assert (train_output == 0).any()
assert torch.equal(eval_output, ones)

## 7. BatchNorm과 LayerNorm

`BatchNorm1d(D)`는 배치의 통계를 사용하므로 batch 구성과 train/eval 모드에 민감합니다. `LayerNorm(D)`는 각 샘플의 마지막 feature 축을 정규화하며 Transformer에서 널리 사용됩니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.BatchNorm1d,nn.LayerNorm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.BatchNorm1d(num_features, eps=1e-5, momentum=0.1, ...)</code></summary>

#### `nn.BatchNorm1d(num_features, eps=1e-5, momentum=0.1, ...)`

- **역할:** mini-batch의 channel별 통계로 1D 또는 sequence feature를 정규화합니다.
- **입력·반환:** feature 수와 설정을 받고 학습·평가 상태를 가진 `nn.Module`을 반환합니다.
- **이 셀에서 쓰는 이유:** activation scale을 안정화하고 running mean·variance를 학습합니다.
- **주의점:** 학습은 batch 통계, 평가는 running 통계를 사용하므로 `train()`·`eval()`이 중요합니다.

##### 관련 수식과 코드 연결

$$
\mu_c=\frac{1}{m}\sum_{i=1}^{m}x_{i,c},\qquad \sigma_c^2=\frac{1}{m}\sum_{i=1}^{m}(x_{i,c}-\mu_c)^2,\qquad y_{i,c}=\gamma_c\frac{x_{i,c}-\mu_c}{\sqrt{\sigma_c^2+\epsilon}}+\beta_c
$$

- **기호:** $c$는 channel, $m$은 batch와 필요하면 공간 위치를 합친 표본 수입니다.
- **수식 → 코드:** 학습 모드는 현재 batch 통계를 쓰고 running 통계를 갱신하며, 평가 모드는 저장된 running 통계를 사용합니다.
- **직관:** channel마다 값의 scale을 맞춘 뒤 학습 가능한 scale과 bias로 표현력을 복원합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `[C]`입니다.
- **공식 문서:** [nn.BatchNorm1d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
normalization_input = torch.randn(8, 6) * 4 + 10  # [B, D]
batch_norm = nn.BatchNorm1d(6)
layer_norm = nn.LayerNorm(6)

batch_normalized = batch_norm(normalization_input)
layer_normalized = layer_norm(normalization_input)

print("BatchNorm feature means:", batch_normalized.mean(dim=0).round(decimals=4))
print("LayerNorm sample means: ", layer_normalized.mean(dim=1).round(decimals=4))
assert batch_normalized.shape == layer_normalized.shape == normalization_input.shape

## 8. 작은 배치 하나를 과적합하라

실제 train loss가 내려가지 않을 때 모델을 키우기 전에 8~32개 샘플만 골라 거의 완벽히 외울 수 있는지 확인합니다. 실패한다면 데이터/label, loss, gradient, optimizer 또는 모델 연결에 버그가 있을 가능성이 큽니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.GELU,F.cross_entropy,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
torch.manual_seed(SEED)
tiny_x = torch.randn(8, 5)
tiny_y = torch.tensor([0, 2, 1, 2, 0, 1, 1, 2])
overfit_model = nn.Sequential(
    nn.Linear(5, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU(), nn.Linear(32, 3)
)
optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.03)

for step in range(201):
    optimizer.zero_grad(set_to_none=True)
    logits = overfit_model(tiny_x)
    loss = nn.functional.cross_entropy(logits, tiny_y)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        accuracy = logits.argmax(-1).eq(tiny_y).float().mean()
        print(f"step={step:3d} loss={loss.item():.5f} accuracy={accuracy.item():.2f}")

final_accuracy = overfit_model(tiny_x).argmax(-1).eq(tiny_y).float().mean()
assert final_accuracy == 1.0

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.linalg.norm,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)</code></summary>

#### `torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)`

- **역할:** 벡터의 크기인 norm을 계산합니다.
- **입력·반환:** 텐서·차수·축을 받고 norm 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 거리, gradient 크기, 정규화 분모를 계산합니다.
- **주의점:** `dim`과 norm 차수에 따라 의미와 반환 shape이 달라집니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [torch.linalg.norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.vector_norm.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# gradient 상태를 한 줄씩 요약하는 도구
overfit_model.zero_grad(set_to_none=True)
diagnostic_loss = nn.functional.cross_entropy(overfit_model(tiny_x), tiny_y)
diagnostic_loss.backward()

gradient_rows = []
for name, parameter in overfit_model.named_parameters():
    gradient_rows.append(
        {
            "parameter": name,
            "shape": tuple(parameter.shape),
            "parameter_norm": float(parameter.detach().norm()),
            "gradient_norm": None
            if parameter.grad is None
            else float(parameter.grad.norm()),
            "finite": parameter.grad is not None
            and bool(torch.isfinite(parameter.grad).all()),
        }
    )
pd.DataFrame(gradient_rows)

<!-- api-explanations:start -->
<!-- api-explanations:keys=clip_grad_norm_ -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)</code></summary>

#### `clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)`

- **역할:** 전체 gradient norm이 한계를 넘으면 같은 비율로 축소합니다.
- **입력·반환:** 파라미터와 최대 norm을 받고 clipping 전 전체 norm을 반환합니다.
- **이 셀에서 쓰는 이유:** RNN·Transformer·RL 학습의 폭주하는 update를 제한합니다.
- **주의점:** `backward()` 뒤 `step()` 전에 호출해야 하며 근본 원인을 없애지는 않습니다.

##### 관련 수식과 코드 연결

$$
g' = g\cdot\min\!\left(1,\frac{c}{\lVert g\rVert_p+\epsilon}\right)
$$

- **기호:** $g$는 모든 parameter gradient를 합친 벡터, $c$는 `max_norm`입니다.
- **수식 → 코드:** 전체 norm이 한계를 넘을 때만 모든 gradient를 같은 비율로 줄입니다.
- **직관:** 방향은 유지하면서 update 크기만 제한합니다.
- **shape 확인:** 각 parameter의 gradient shape은 유지됩니다.
- **공식 문서:** [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# Gradient clipping은 폭발한 gradient의 방향은 유지하고 전체 norm을 제한합니다.
clip_model = nn.Linear(3, 2)
huge_input = torch.randn(16, 3) * 10_000
clip_model(huge_input).square().mean().backward()

before_clip = torch.sqrt(sum(p.grad.square().sum() for p in clip_model.parameters()))
returned_norm = nn.utils.clip_grad_norm_(clip_model.parameters(), max_norm=1.0)
after_clip = torch.sqrt(sum(p.grad.square().sum() for p in clip_model.parameters()))
print(
    "before/returned/after:",
    before_clip.item(),
    float(returned_norm),
    after_clip.item(),
)
assert after_clip <= 1.0001

## 따라 치기 TODO + 마무리

새 셀에서 다음 실험을 진행하세요.

1. 선형 회귀의 learning rate를 `0.001, 0.08, 1.0`으로 바꾸고 loss 곡선의 차이를 기록합니다.
2. XOR MLP에서 `Tanh`를 제거해 다시 학습하고 결과를 설명합니다.
3. 10층 MLP를 만들어 initialization별로 각 층 activation std를 수집합니다.
4. 작은 배치 과적합 모델에 dropout `p=0.9`를 넣고 왜 디버깅이 어려워지는지 확인합니다.
5. `BatchNorm1d`를 batch size 1로 학습할 때 발생하는 문제를 관찰하고 `LayerNorm`과 비교합니다.
6. overfit test가 통과한 뒤 train/validation split으로 확장하고 일반화 격차를 측정합니다.

기억할 핵심: **복잡한 모델을 고치려 하지 말고, 먼저 작은 데이터에서 forward → loss → gradient → update가 모두 연결됐음을 증명하세요.**